# Questão 6 - Dimensão de calendário

***Tarefas:***
- Construa uma dimensão de datas utilizando sql
- Cruze a dimensão de datas com a tabela de vendas para análise (não esqueça de considerar os dias sem vendas).

***Objetivo***
- Descobrir: "Qual é o dia da semana (Segunda, Terça...) que temos a pior média de vendas?" para decidir se vale a pena fechar a loja nesses dias. 

### 1.Preparação do Ambiente e Limpeza de Dados

In [1]:
#importar bibliotecas
import pandas as pd
import sqlite3

In [2]:
# 1. Carregar as vendas
df_vendas = pd.read_csv('datasets/vendas_2023_2024.csv')
df_vendas

,Unnamed: 0.1,Unnamed: 0,id,id_client,id_product,qtd,total,sale_date,sale_date_dt
0,0,0,0,42,105,11,3405.00,2023-09-10,2023-09-10
1,1,1,1,3,136,9,16873.90,15-09-2024,2024-09-15
2,2,2,2,25,139,7,9475.30,2024-08-13,2024-08-13
3,3,3,4,20,23,5,55893.00,2023-02-03,2023-02-03
4,4,4,5,8,57,4,451403.90,2024-02-12,2024-02-12
...,...,...,...,...,...,...,...,...,...
9890,9890,9890,9995,30,139,6,8549.00,11-03-2023,2023-11-03
9891,9891,9891,9996,9,111,7,28497.15,17-09-2023,2023-09-17
9892,9892,9892,9997,38,123,2,5276.30,20-06-2023,2023-06-20
9893,9893,9893,9998,33,97,6,771409.50,23-10-2024,2024-10-23


In [3]:

# 2. CONVERSÃO ESSENCIAL: 
# O parâmetro format='mixed' é obrigatório aqui para lidar com os dois padrões 
# identificados (YYYY-MM-DD e DD-MM-YYYY) [2].
df_vendas['sale_date_clean'] = pd.to_datetime(df_vendas['sale_date'], format='mixed', dayfirst=False)

# 3. AGORA SIM: Normalizar para string ISO para o SQLite
# Como agora a coluna é do tipo datetime, o .dt.strftime funcionará sem erros.
df_vendas['sale_date_clean'] = df_vendas['sale_date_clean'].dt.strftime('%Y-%m-%d')


In [4]:
# 4. Verificação rápida no Pandas antes do to_sql
print(df_vendas['sale_date_clean'].isnull().sum())

0


In [5]:
# 5. Criar conexão SQL em memória
conn = sqlite3.connect(':memory:')
df_vendas.to_sql('vendas', conn, index=False, if_exists='replace')

print("Dados de vendas carregados e datas padronizadas.")

Dados de vendas carregados e datas padronizadas.


### 2. Questão 6.1 — Código SQL 

In [6]:
# 6. Consulta SQL consolidando o Calendário e as Vendas (Questão 6.1)
query_analise_dias = """
WITH RECURSIVE Dim_Datas(data) AS (
    -- Gera todas as datas entre o início e o fim do registro de vendas [4]
    SELECT MIN(sale_date_clean) FROM vendas
    UNION ALL
    SELECT date(data, '+1 day') FROM Dim_Datas
    WHERE data < (SELECT MAX(sale_date_clean) FROM vendas)
),
Vendas_Agregadas_Dia AS (
    -- Soma o faturamento total por dia real [4]
    SELECT sale_date_clean, SUM(total) as soma_total_dia
    FROM vendas
    GROUP BY sale_date_clean
),
Calendario_Integrado AS (
    -- Une o calendário com as vendas e força o valor zero onde não houve transação [5, 6]
    SELECT 
        d.data,
        COALESCE(v.soma_total_dia, 0) as valor_venda,
        CASE strftime('%w', d.data)
            WHEN '0' THEN 'Domingo'
            WHEN '1' THEN 'Segunda-feira'
            WHEN '2' THEN 'Terça-feira'
            WHEN '3' THEN 'Quarta-feira'
            WHEN '4' THEN 'Quinta-feira'
            WHEN '5' THEN 'Sexta-feira'
            WHEN '6' THEN 'Sábado'
        END AS dia_semana_pt
    FROM Dim_Datas d
    LEFT JOIN Vendas_Agregadas_Dia v ON d.data = v.sale_date_clean
)
-- Calcula a média real considerando a inatividade da loja [5]
SELECT 
    dia_semana_pt,
    ROUND(AVG(valor_venda), 2) AS media_vendas_real
FROM Calendario_Integrado
GROUP BY dia_semana_pt
ORDER BY media_vendas_real ASC;
"""

In [7]:
df_resultado_dias = pd.read_sql(query_analise_dias, conn)
df_resultado_dias

,dia_semana_pt,media_vendas_real
0,Segunda-feira,3285975.58
1,Domingo,3366781.09
2,Quinta-feira,3560174.45
3,Quarta-feira,3649345.28
4,Sexta-feira,3651962.65
5,Sábado,3667602.92
6,Terça-feira,3816335.14


### 3. Validação do dia da semana com menor média de vendas (Questão 6.2)

In [8]:
df_resultado_dias.iloc[df_resultado_dias.idxmin()]

,dia_semana_pt,media_vendas_real
1,Domingo,3366781.09
0,Segunda-feira,3285975.58


In [9]:
df_resultado_dias.describe()

,media_vendas_real
count,7.000000e+00
mean,3.571168e+06
std,1.849665e+05
min,3.285976e+06
25%,3.463478e+06
50%,3.649345e+06
75%,3.659783e+06
max,3.816335e+06


In [10]:
df_resultado_dias["media_vendas_real"].min()

3285975.58

### Relatório final (Questão 6.3)

***Explique:***

- Por que é necessário utilizar uma tabela de datas (calendário) em vez de agrupar diretamente a tabela de vendas? O arquivo vendas_2023_2024.csv registra apenas as transações que efetivamente ocorreram
. Dias em que a loja abriu mas não vendeu nada (faturamento zero) não possuem linhas na tabela. Se agruparmos diretamente as vendas, o cálculo da média dividirá o faturamento total apenas pelos dias "com sorte", ignorando os dias de inatividade. O uso de uma tabela de datas (dimensão calendário) garante que todos os dias do período sejam incluídos no denominador da média, transformando o "feeling" em faturamento consolidado real.
- O que aconteceria com a média de vendas se um dia da semana tivesse muitos dias sem nenhuma venda registrada?R$ 5.000,00 em apenas uma delas e zero nas outras nove, a análise direta diria que a média é de R$ 5.000,00. Ao utilizar o calendário, as nove terças-feiras com valor zero são incluídas via LEFT JOIN e COALESCE, revelando que a média real é de apenas R$ 500,00. Isso permite ao Sr. Almir identificar com precisão os dias de pior desempenho para otimizar os custos operacionais da loja física.

***Explique:***

- Por que é necessário utilizar uma tabela de datas (calendário) em vez de agrupar diretamente a tabela de vendas? O arquivo vendas_2023_2024.csv registra apenas as transações que efetivamente ocorreram
. Dias em que a loja abriu mas não vendeu nada (faturamento zero) não possuem linhas na tabela. Se agruparmos diretamente as vendas, o cálculo da média dividirá o faturamento total apenas pelos dias "com sorte", ignorando os dias de inatividade. O uso de uma tabela de datas (dimensão calendário) garante que todos os dias do período sejam incluídos no denominador da média, transformando o "feeling" em faturamento consolidado real.
- O que aconteceria com a média de vendas se um dia da semana tivesse muitos dias sem nenhuma venda registrada? Se tivessemos 5.000,00 reais em apenas uma delas e zero nas outras nove, a análise direta diria que a média é de 5.000,00 reais, levando a uma conclusão falsa. Ao utilizar o calendário, as nove datas com valor zero são incluídas via LEFT JOIN e COALESCE, revelando que a média real seria de apenas R$ 500,00. Isso permite ao Sr. Almir identificar com precisão os dias de pior desempenho para otimizar os custos operacionais da loja física. 